## IMPORTS


In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.impute import SimpleImputer
import os
from imblearn.pipeline import Pipeline 
import xgboost as  xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import sys
from sklearn.experimental import enable_iterative_imputer
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.impute import IterativeImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
import joblib
import logging
import optuna

ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper



c:\Users\gonzalo.iglesias\AppData\Local\miniconda3\envs\machinelearning2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOS OUTPUTS DE OPTUNA QUE LOS IMPRIMA EN UN TXT PARA SIN VOLVERLO A CORRER

In [2]:
log_file = logging.FileHandler("optuna_resultados.txt")
log_file.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
optuna.logging.get_logger("optuna").addHandler(log_file)

In [3]:
TRAIN = True
SAVE_MODEL = True


## LOAD DATA + SAVE FUCTION


In [4]:
X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv")
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv")



target_folder = '../comparations'

def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    # Point the file to inside the folder
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")


## XGBOOST

In [5]:
num_classes = int(y_train['customer_segment'].nunique())
modelo_xgb = xgb.XGBClassifier(
    objective='multi:softmax', 
    num_class=num_classes, 
    random_state=42,
    eval_metric='mlogloss'
)

## DEFINIMOS LOS RANGOS Y CATEGORIAS QUE QUEREMOS QUE OPTUNA OPTIMICE

In [6]:
imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}


for method_name, sampler in imbalance_methods.items():
    print(f"\n--- {method_name} ---")

    

    if sampler is None:
        pipe = Pipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('classifier', modelo_xgb)])
    else:
        pipe = Pipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('sampler', sampler), ('classifier', modelo_xgb)])

    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall    = recall_score(y_test, y_pred, average='weighted')
    f1        = f1_score(y_test, y_pred, average='weighted')
    roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

    print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

    save_experiment(
        model_name="XgBoost",
        imbalance_method=method_name,
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1=f1,
        roc_auc=roc_auc
    )


--- Baseline ---
Accuracy: 0.7531 | F1: 0.7517 | ROC-AUC: 0.9009
✅ Success! Results for XgBoost + Baseline saved in the '../comparations' folder.

--- RandomUnderSampler ---
Accuracy: 0.7276 | F1: 0.7313 | ROC-AUC: 0.8928
✅ Success! Results for XgBoost + RandomUnderSampler saved in the '../comparations' folder.

--- TomekLinks ---
Accuracy: 0.7564 | F1: 0.7545 | ROC-AUC: 0.9018
✅ Success! Results for XgBoost + TomekLinks saved in the '../comparations' folder.

--- RandomOverSampler ---
Accuracy: 0.7425 | F1: 0.7455 | ROC-AUC: 0.8988
✅ Success! Results for XgBoost + RandomOverSampler saved in the '../comparations' folder.

--- SMOTE ---
Accuracy: 0.7501 | F1: 0.7512 | ROC-AUC: 0.8989
✅ Success! Results for XgBoost + SMOTE saved in the '../comparations' folder.

--- ADASYN ---
Accuracy: 0.7442 | F1: 0.7448 | ROC-AUC: 0.8983
✅ Success! Results for XgBoost + ADASYN saved in the '../comparations' folder.

--- SMOTETomek ---
Accuracy: 0.7509 | F1: 0.7521 | ROC-AUC: 0.8992
✅ Success! Results

In [ ]:
def objective(trial):
    

    # ESCOGER TIPO DE MUESTREO
    nombre_metodo = "TomekLinks" 
    metodo_elegido = TomekLinks()
    # ESCOGER LOS HIPERPARÁMETROS DE XGBOOST
    param_n_estimators = trial.suggest_int('n_estimators', 50, 1000)
    param_max_depth = trial.suggest_int('max_depth', 3, 8)  
    param_learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3, log=True)
    param_gamma = trial.suggest_float('gamma', 0.0, 5.0 )
    param_lambda = trial.suggest_float('lambda', 0.5, 10.0, log=True) 
    param_alpha = trial.suggest_float('alpha', 0.0, 5.0)
    param_subsample = trial.suggest_float('subsample',        0.5, 1.0)
    param_colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    param_min_child_weight = trial.suggest_int(  'min_child_weight', 1, 20)  # 
    param_max_delta_step = trial.suggest_float('max_delta_step', 0.0, 10.0)

    param_scale_pos_weight = 1.0  

    modelo_xgb = xgb.XGBClassifier(
        n_estimators = param_n_estimators,
        max_depth = param_max_depth,
        learning_rate = param_learning_rate,
        gamma = param_gamma,
        reg_lambda = param_lambda,
        reg_alpha = param_alpha,
        subsample = param_subsample,
        colsample_bytree = param_colsample_bytree,
        min_child_weight = param_min_child_weight,
        scale_pos_weight = param_scale_pos_weight,
        max_delta_step = param_max_delta_step,
        objective = 'multi:softmax',
        eval_metric = 'mlogloss',
        num_class = len(np.unique(y_train)),
        random_state = 42,
        n_jobs = -1
    )

    pasos_pipeline = [
        ('capping_outliers', IQRCapper(factor=1.5)),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('sampler', metodo_elegido),
        ('clf', modelo_xgb)
    ]

    


    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        pasos_pipeline,
        X_train, y_train,
        cv=cv,
        scoring='f1_macro',
        n_jobs=1
    )

    return np.mean(scores)
            
  

EJECUTAMOS OPTUNA

In [ ]:
if TRAIN: 
    study = optuna.create_study(direction='maximize', study_name="Optimizacion_XGB_Balanceo")
    study.optimize(objective, n_trials=500)
    print(f"Mejor F1-Score Macro: {study.best_value:.4f}")
    print("Mejor combinación encontrada:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")

    with open("mejores_parametros_XGBoost.txt", "w") as archivo:
        archivo.write("Mejores hiperparámetros encontrados para XGBoost:\n")
        archivo.write("="*40 + "\n")
        for key, value in study.best_params.items():
            archivo.write(f"  - {key}: {value}\n")

    print("¡Parámetros guardados correctamente en 'mejores_parametros.txt'!")
    
    


GUARDAMOS EL MEJOR MODELO

In [ ]:
if TRAIN:
    print("Reconstruyendo y entrenando el modelo final con toda la data...")
    
    mejor_metodo_nombre = "TomekLinks"
    mejor_metodo = TomekLinks()

    mejor_xgb = xgb.XGBClassifier(
        n_estimators=study.best_params['n_estimators'],
        max_depth=study.best_params['max_depth'],
        learning_rate=study.best_params['learning_rate'],
        subsample=study.best_params['subsample'],
        colsample_bytree=study.best_params['colsample_bytree'],
        objective='multi:softmax',
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )

    pasos_finales = [
        ('capping_outliers', IQRCapper(factor=1.5)),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('sampler', mejor_metodo),
        ('clf', mejor_xgb)
    ]



    pasos_finales.fit(X_train, y_train)
    if SAVE_MODEL:
        joblib.dump(pasos_finales, 'modelo_xgboost_ganador.pkl')
        print("¡Modelo final entrenado y guardado correctamente como 'modelo_xgboost_ganador.pkl'!")
        


In [ ]:
pipe = joblib.load('modelo_xgboost_ganador.pkl')

y_pred       = pipe.predict(X_test)
y_pred_proba = pipe.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba, 
                           multi_class='ovr',  
                           average='weighted')  

new_result = {
    'Model': ["XGBoost"],
    'Imbalance_Method': ["None"],
    'Accuracy': [round(accuracy,  4)],
    'Precision': [round(precision, 4)],
    'Recall': [round(recall,    4)],
    'F1_Score': [round(f1,        4)],
    'ROC_AUC': [round(roc_auc,   4)]
}

print(new_result)



c:\Users\gonzalo.iglesias\AppData\Local\miniconda3\envs\machinelearning2\Lib\site-packages\xgboost\core.py:748: FutureWarning: Pass `objective` as keyword args.
  warnings.warn(msg, FutureWarning)


{'objective': 'multi:softmax', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'mlogloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': None, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': None, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None, 'num_class': 4}
{'Model': ['XGBoost'], 'Imbalance_Method': ['None'], 'Acc

## STACKING CLASIFFIER

In [ ]:
"""
modelos_base = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('xgb', xgb.XGBClassifier( 
        objective='multi:softprob', 
        num_class=3, 
        eval_metric='mlogloss',
        random_state=42
    ))
]

meta_modelo = LogisticRegression(max_iter=1000)

stacking_clf = StackingClassifier(
    estimators=modelos_base,
    final_estimator=meta_modelo,
    cv=5, 
    n_jobs=-1
)

mi_pipeline_stacking = Pipeline(steps=[
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()), 
    ('stacking', stacking_clf)
])


mi_pipeline_stacking.fit(X_train, np.ravel(y_train)) 

predicciones = mi_pipeline_stacking.predict(X_test)
probabilidades = mi_pipeline_stacking.predict_proba(X_test)

y_test1d = np.ravel(y_test)




exactitud = accuracy_score(y_test1d, predicciones)
print(f"Accuracy Global: {exactitud:.4f}")

# Calculamos F1-Score (usamos 'weighted' para tener en cuenta el desbalanceo)
f1_ponderado = f1_score(y_test1d, predicciones, average='weighted')
print(f"F1-Score (Weighted): {f1_ponderado:.4f}")


"""
